# Functional annotation and Gene Ontology enrichment analysis

This notebook contains the functional annotation and Gene Ontology (GO) enrichment workflow for *Leptinotarsa decemlineata*.

The workflow integrates NCBI RefSeq, InterProScan, and ProteInfer annotations, constructs extended gene-to-GO annotation tables, loads differential expression results generated in the preceding DESeq2 analysis, and performs GO enrichment analysis for haemocytes and fat body.

InterProScan and ProteInfer input tables were generated separately in the preceding annotation step from the predicted protein sequences of the Ldec_2.0 genome assembly (GCF_000500325.1).


In [ ]:
import gc, re, os
import pandas as pd
import numpy as np
import collections as cx
import warnings

In [ ]:
from Bio import Entrez
from Bio import SeqIO
from time import sleep
from datetime import date

In [ ]:
## -- GOenrich - is our current favorite
## https://pypi.org/project/goenrich/
## https://github.com/jdrudolph/goenrich

## -- GOATOOLS
## https://github.com/tanghaibao/goatools

## some ideas for plotting:
## https://bmcbioinformatics.biomedcentral.com/articles/10.1186/s12859-022-04828-2
## https://notebook.community/tanghaibao/goatools/notebooks/goea_nbt3102 -- goeaobj.run_study( ... , prt = None)

In [ ]:
import csv, glob, io
import gff3_parser

import goatools
from goatools.obo_parser import GODag
from __future__ import print_function
from goatools.anno.genetogo_reader import Gene2GoReader
from goatools.goea.go_enrichment_ns import GOEnrichmentStudyNS, GOEnrichmentStudy

import goenrich

In [ ]:
warnings.filterwarnings('ignore')

In [ ]:
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', 500)
#pd.set_option('display.max_colwidth', 150)

In [ ]:
gc.collect()

In [ ]:
## https://www.ncbi.nlm.nih.gov/genome/annotation_euk/Leptinotarsa_decemlineata/100/
## https://ftp.ncbi.nlm.nih.gov/genomes/all/annotation_releases/7539/100/GCF_000500325.1_Ldec_2.0/
## goann = pd.read_csv('GCF_000500325.1_Ldec_2.0_gene_ontology.tab', sep = '\t')

In [ ]:
## goann.GeneID.nunique() ## 10633, 6913 genes were annotated with KEGG data

In [ ]:
## goann.GO_ID.nunique() - 4730

In [ ]:
## goann.groupby('GeneID')['GO_ID'].nunique().describe()

In [ ]:
## installed python Packages for GO-analysis:
## https://github.com/tanghaibao/goatools
## https://github.com/zqfang/GSEApy
## https://github.com/jdrudolph/goenrich
## https://github.com/kylewellband/GOenrich

## 1. Gene Ontology reference files


In [ ]:
dag = GODag('./GOdb/go-basic-1.4.obo', optional_attrs=['def']) ## read the basic GO onthology

In [ ]:
dag_old = GODag('./GOdb/go-basic.obo', optional_attrs=['def']) ## read the basic GO onthology

In [ ]:
## error! dag_plus = GODag('./GOdb/go-plus.owl', optional_attrs=['def'])

In [ ]:
slim_dag = GODag('./GOdb/goslim_generic.obo', optional_attrs=['def']) ## read the GO slim onthology

In [ ]:
## https://geneontology.org/docs/download-ontology/ -- download here

## 2. InterProScan and ProteInfer annotations


In [ ]:
protinfer = pd.read_csv('./GOdb/protinfer.csv', sep ='\t')

In [ ]:
protinfer.head(5)

In [ ]:
interpro = pd.read_csv('./GOdb/interpro.csv', sep ='\t')

In [ ]:
interpro.tail(5)

In [ ]:
### read the NCBI GO annotation for multiple species
all_ncbi_gene2go = pd.read_csv('./GOdb/gene2go', sep = '\t')

In [ ]:
## extract only Ldec data
ldec_goann = all_ncbi_gene2go[ all_ncbi_gene2go['#tax_id'].eq(7539) ]
del all_ncbi_gene2go
gc.collect()

In [ ]:
#### prepare the ProtInfer table

In [ ]:
protinfer.loc[:, 'GeneID']= protinfer.Gene.str.replace('LOC','').astype(int)

In [ ]:
## extract only the GO annotation
protinfer_go = protinfer[protinfer.predicted_label.str.startswith('GO')]

In [ ]:
protinfer_go.loc[:, 'GO_ID'] = protinfer_go.predicted_label.values
protinfer_go.loc[:, 'GO_term'] = protinfer_go.description.values
protinfer_go.loc[:, '#tax_id'] = 7539
protinfer_go.loc[:, 'Evidence'] = 'ProtInfer'

In [ ]:
protinfer_go.GO_ID.nunique(), ldec_goann.GO_ID.nunique(), len( set( protinfer_go.GO_ID.unique() ).intersection(
    set( ldec_goann.GO_ID.unique() ) ) ) 

In [ ]:
#### prepare the InterPro table

In [ ]:
interpro['GeneID']= interpro.Gene.str.replace('LOC','').astype(int)
interpro['GO_ID'] = interpro.Main_GO.values
interpro['GO_term'] = interpro.GO_Annotations.values
interpro['#tax_id'] = 7539
interpro['Evidence'] = 'InterPro'

In [ ]:
interpro.GO_ID.nunique(), ldec_goann.GO_ID.nunique(), len( set( interpro.GO_ID.unique() ).intersection(
    set( ldec_goann.GO_ID.unique() ) ) ) ## so, the annotation by InterPro is quite small and brings little new information

In [ ]:
protinfer_go.GO_ID.nunique(), ldec_goann.GO_ID.nunique(), len( set( protinfer_go.GO_ID.unique() ).intersection(
    set( ldec_goann.GO_ID.unique() ) ) ) ## while the ProtInfer almost doubles the number of GO-terms

In [ ]:
protinfer_go_hq = protinfer_go[protinfer_go.confidence.ge(.9)] # select only GO predictions with confidence >= 0.9

In [ ]:
protinfer_go_hq.nunique(), protinfer_go.nunique() ## check for unique values counts in columns

In [ ]:
## merge all Ldec GO-annotations into one
ldec_goann_cmb = ldec_goann.merge( protinfer_go, how = 'outer').merge( interpro, how = 'outer' ).iloc[:,:8]
## merge all Ldec GO-annotations into one, using protinfer_go_hq
ldec_goann_cmb_pihq = ldec_goann.merge( protinfer_go_hq, how = 'outer').merge( interpro, how = 'outer' ).iloc[:,:8]

In [ ]:
ugos = ldec_goann_cmb.GO_ID.unique() ## get all unique GO IDs

In [ ]:
## unify Categories, Terms and Qualifiers
## a dictionary of GO_IDs to functional category
go2cat = dict( zip( ugos, map( lambda x: dag.get(x).namespace if type( dag.get(x) ) == goatools.obo_parser.GOTerm else '', 
              ugos ) ) )
## a dictionary of GO_IDs to Terms
go2term = dict( zip( ugos, map( lambda x: dag.get(x).name if type( dag.get(x) ) == goatools.obo_parser.GOTerm else '', 
              ugos ) ) )

ldec_goann_cmb['Category'] = ldec_goann_cmb.GO_ID.apply( lambda x: go2cat[x] ).replace( ## map categories by GO ID with go2cat and apply
                         { 'molecular_function' : 'Function', 
                           'biological_process' : 'Process', 
                           'cellular_component' : 'Component' , '':''} ) ## replace the values with replace
ldec_goann_cmb_pihq['Category'] = ldec_goann_cmb_pihq.GO_ID.apply( lambda x: go2cat[x] ).replace( 
                         { 'molecular_function' : 'Function', 
                           'biological_process' : 'Process', 
                           'cellular_component' : 'Component' , '':''} )

ldec_goann_cmb['Qualifier'] = ldec_goann_cmb.Category.apply( lambda x: {'Function':'enables',
                            'Component':'located_in',
                            'Process':'involved_in',
                            '':''}[x] )
ldec_goann_cmb_pihq['Qualifier'] = ldec_goann_cmb_pihq.Category.apply( lambda x: {'Function':'enables',
                            'Component':'located_in',
                            'Process':'involved_in',
                            '':''}[x] )

ldec_goann_cmb['GO_term'] = ldec_goann_cmb.GO_ID.apply( lambda x: go2term[x] ) ## unify the GO terms
ldec_goann_cmb_pihq['GO_term'] = ldec_goann_cmb_pihq.GO_ID.apply( lambda x: go2term[x] )

ldec_goann_cmb = ldec_goann_cmb[ ~ldec_goann_cmb.GO_term.eq('') ]
ldec_goann_cmb_pihq = ldec_goann_cmb_pihq[ ~ldec_goann_cmb_pihq.GO_term.eq('') ]

In [ ]:
ldec_goann_protinf = ldec_goann_cmb[ ~ldec_goann_cmb.Evidence.eq('InterPro') ].copy()
ldec_goann_protinf_only = ldec_goann_cmb[ ldec_goann_cmb.Evidence.eq('ProtInfer') ].copy()

ldec_goann_pihq_protinf = ldec_goann_cmb_pihq[ ~ldec_goann_cmb_pihq.Evidence.eq('InterPro') ].copy()
ldec_goann_pihq_protinf_only = ldec_goann_cmb_pihq[ ldec_goann_cmb_pihq.Evidence.eq('ProtInfer') ].copy()

In [ ]:
ldec_goann_cmb.Evidence = 'IEA'
ldec_goann_protinf.Evidence = 'IEA'
ldec_goann_protinf_only.Evidence = 'IEA'
ldec_goann_cmb_pihq.Evidence = 'IEA'
ldec_goann_pihq_protinf.Evidence = 'IEA'
ldec_goann_pihq_protinf_only.Evidence = 'IEA'

In [ ]:
ldec_goann_cmb = ldec_goann_cmb.groupby(['GeneID','GO_ID']).head(1)
ldec_goann_protinf = ldec_goann_protinf.groupby(['GeneID','GO_ID']).head(1)
ldec_goann_protinf_only = ldec_goann_protinf_only.groupby(['GeneID','GO_ID']).head(1)
ldec_goann_cmb_pihq = ldec_goann_cmb_pihq.groupby(['GeneID','GO_ID']).head(1)
ldec_goann_pihq_protinf = ldec_goann_pihq_protinf.groupby(['GeneID','GO_ID']).head(1)
ldec_goann_pihq_protinf_only = ldec_goann_pihq_protinf_only.groupby(['GeneID','GO_ID']).head(1)

In [ ]:
## calculate the number of unique values in columns and a number of unique "gene"-"GO" combinations
pd.concat( [ldec_goann_cmb.nunique(), pd.Series( {'N': ldec_goann_cmb[['GeneID','GO_ID']].drop_duplicates().shape[0]} ) ] ).to_frame().T

In [ ]:
pd.concat( [ldec_goann_protinf.nunique(), pd.Series( {'N': ldec_goann_protinf[['GeneID','GO_ID']].drop_duplicates().shape[0]} ) ] ).to_frame().T

In [ ]:
pd.concat( [ldec_goann_protinf_only.nunique(), 
            pd.Series( {'N': ldec_goann_protinf_only[['GeneID','GO_ID']].drop_duplicates().shape[0]} ) ] ).to_frame().T

In [ ]:
pd.concat( [ldec_goann_pihq_protinf.nunique(), 
            pd.Series( {'N': ldec_goann_pihq_protinf[['GeneID','GO_ID']].drop_duplicates().shape[0]} ) ] ).to_frame().T

In [ ]:
pd.concat( [ldec_goann_pihq_protinf_only.nunique(), 
            pd.Series( {'N': ldec_goann_pihq_protinf_only[['GeneID','GO_ID']].drop_duplicates().shape[0]} ) ] ).to_frame().T

## 3. Integration and export of gene-to-GO annotations


In [ ]:
## GO annotation table with NCBI, InterPro and ProtInfer annotations
ldec_goann_cmb.to_csv('./GOdb/Ldec_gene2go_ncbi_ip_pi', sep = '\t', index = False) 
## GO annotation table with NCBI and ProtInfer annotations
ldec_goann_protinf.to_csv('./GOdb/Ldec_gene2go_ncbi_pi', sep = '\t', index = False)
## GO annotation table with ProtInfer annotation only
ldec_goann_protinf_only.to_csv('./GOdb/Ldec_gene2go_pi', sep = '\t', index = False)
## GO annotation table with NCBI, InterPro and highly confident ProtInfer annotations
ldec_goann_cmb_pihq.to_csv('./GOdb/Ldec_gene2go_ncbi_ip_pihq', sep = '\t', index = False)
## GO annotation table with NCBI and highly confident ProtInfer annotations
ldec_goann_pihq_protinf.to_csv('./GOdb/Ldec_gene2go_ncbi_pihq', sep = '\t', index = False)
## GO annotation table with highly confident ProtInfer annotation only
ldec_goann_pihq_protinf_only.to_csv('./GOdb/Ldec_gene2go_pihq', sep = '\t', index = False)

In [ ]:
gc.collect()

## 4. Ldec_2.0 genome annotation and background gene set


In [ ]:
ldec_gff = gff3_parser.parse_gff3('./Ldec_2.0/GCF_000500325.1_Ldec_2.0_genomic.gff', verbose = False,
                                  parse_attributes = True )

In [ ]:
genes = ldec_gff[ldec_gff.Type.eq('gene')]
cdss  = ldec_gff[ldec_gff.Type.eq('CDS')]

In [ ]:
#all_genes = ldec_goann_cmb2.GeneID.unique().tolist()
all_genes = list ( set( genes.Dbxref.str.replace('^GeneID:','', regex = True).astype(int) ) )

## 5. Differential expression results


In [ ]:
deg = pd.read_excel('./DEG_ldec_DESeq2.xlsx', sheet_name = None)
deg2 = pd.read_excel('./DEG_ldec_DESeq2_bbas_vs_mrob.xlsx', sheet_name = None)

In [ ]:
gc.collect()

In [ ]:
deg.keys()

In [ ]:
deg2.keys()

In [ ]:
bbas_hae = deg['Bbas_Hae']
bbas_hae = bbas_hae[ bbas_hae.gene_id.str.contains('^gene-LOC', regex = True) ]
mrob_hae = deg['Mrob_Hae']
mrob_hae = mrob_hae[ mrob_hae.gene_id.str.contains('^gene-LOC', regex = True) ]
bbas_fat = deg['Bbas_Fat_woC17']
bbas_fat = bbas_fat[ bbas_fat.gene_id.str.contains('^gene-LOC', regex = True) ]
mrob_fat = deg['Mrob_Fat_woC17']
mrob_fat = mrob_fat[ mrob_fat.gene_id.str.contains('^gene-LOC', regex = True) ]
bbas_mrob_hae = deg2['Bbas_vs_Mrob_Hae']
bbas_mrob_hae = bbas_mrob_hae[ bbas_mrob_hae.gene_id.str.contains('^gene-LOC', regex = True) ]
bbas_mrob_fat = deg2['Bbas_vs_Mrob_Fat']
bbas_mrob_fat = bbas_mrob_fat[ bbas_mrob_fat.gene_id.str.contains('^gene-LOC', regex = True) ]

In [ ]:
bbas_hae.loc[:,'treatment'] = 'Bbas_vs_Ctrl'
mrob_hae.loc[:,'treatment'] = 'Mrob_vs_Ctrl'
bbas_fat.loc[:,'treatment'] = 'Bbas_vs_Ctrl'
mrob_fat.loc[:,'treatment'] = 'Mrob_vs_Ctrl'
bbas_mrob_hae.loc[:,'treatment'] = 'Mrob_vs_Bbas'
bbas_mrob_fat.loc[:,'treatment'] = 'Mrob_vs_Bbas'

bbas_hae.loc[:,'tissue'] = 'Haemocytes'
mrob_hae.loc[:,'tissue'] = 'Haemocytes'
bbas_fat.loc[:,'tissue'] = 'Fat body'
mrob_fat.loc[:,'tissue'] = 'Fat body'
bbas_mrob_hae.loc[:,'tissue'] = 'Haemocytes'
bbas_mrob_fat.loc[:,'tissue'] = 'Fat body'

In [ ]:
bbas_hae.loc[:,'GeneID'] = bbas_hae.gene_id.str.replace('^gene-LOC','', regex = True).astype(int)
mrob_hae.loc[:,'GeneID'] = mrob_hae.gene_id.str.replace('^gene-LOC','', regex = True).astype(int)
bbas_fat.loc[:,'GeneID'] = bbas_fat.gene_id.str.replace('^gene-LOC','', regex = True).astype(int)
mrob_fat.loc[:,'GeneID'] = mrob_fat.gene_id.str.replace('^gene-LOC','', regex = True).astype(int)
bbas_mrob_hae.loc[:,'GeneID'] = bbas_mrob_hae.gene_id.str.replace('^gene-LOC','', regex = True).astype(int)
bbas_mrob_fat.loc[:,'GeneID'] = bbas_mrob_fat.gene_id.str.replace('^gene-LOC','', regex = True).astype(int)

In [ ]:
bbas_mrob_hae.head(5)

## 6. Gene Ontology enrichment analysis

GO enrichment is evaluated for the differential-expression comparisons using the extended *L. decemlineata* gene-to-GO annotations.


### 6.1 GOATOOLS


In [ ]:
## make directory GOATools_out
## make subdirectories:
## - bbas_hae
## - mrob_hae
## - bbas_fat
## - mrob_fat
## - bbas_mrob_hae
## - bbas_mrob_fat
os.mkdir('./GOEAnalysis')
os.mkdir('./GOEAnalysis/GOATools_out')
for oo in [ 'bbas_hae', 'mrob_hae', 'bbas_fat', 'mrob_fat', 'bbas_mrob_hae', 'bbas_mrob_fat' ]:
    os.mkdir(f'./GOEAnalysis/GOATools_out/{oo}')


In [ ]:
%%capture
## %%capture to suppress the messy printing output
for annotab in ['Ldec_gene2go_ncbi_ip_pi', 'Ldec_gene2go_ncbi_pi', 'Ldec_gene2go_pi', 
                'Ldec_gene2go_ncbi_ip_pihq', 'Ldec_gene2go_ncbi_pihq', 'Ldec_gene2go_pihq']:
    # Read NCBI's gene2go. Store annotations in a list of namedtuples
    objanno = Gene2GoReader(f'./GOdb/{annotab}', taxids=[7539])
    # Get namespace2association:
    ns2assoc = objanno.get_ns2assc()
    for deg_name in [ 'bbas_hae', 'mrob_hae', 'bbas_fat', 'mrob_fat', 'bbas_mrob_hae', 'bbas_mrob_fat' ]:
        deg_tab = globals()[ deg_name ]
        gene_query      = deg_tab[ (deg_tab.padj < 0.1) ].GeneID.values.tolist()
        gene_query_15   = deg_tab[ (deg_tab.log2FoldChange.abs() >= .5) & (deg_tab.padj < 0.1) ].GeneID.values.tolist()
        gene_query_up   = deg_tab[ (deg_tab.log2FoldChange >= .5) & (deg_tab.padj < 0.1) ].GeneID.values.tolist()
        gene_query_down = deg_tab[ (deg_tab.log2FoldChange <= -.5) & (deg_tab.padj < 0.1) ].GeneID.values.tolist()
        
        goeaobj = GOEnrichmentStudyNS( all_genes, # List of Ldec genes
                                       ns2assoc, # geneid/GO associations
                                       dag, # Ontologies
                                       propagate_counts = False,
                                       alpha = 0.05, # default significance cut-off
                                       methods = ['fdr_bh']) # defult multipletest correction method        
        
        goea_results_all  = goeaobj.run_study( gene_query, prt = None )
        goea_results_down = goeaobj.run_study( gene_query_down , prt = None )
        goea_results_up   = goeaobj.run_study( gene_query_up , prt = None )
        goea_results_15   = goeaobj.run_study( gene_query_15 , prt = None)

        goea_results_all  = [r for r in goea_results_all if r.p_fdr_bh < 0.1]
        goea_results_down = [r for r in goea_results_down if r.p_fdr_bh < 0.1]
        goea_results_up   = [r for r in goea_results_up if r.p_fdr_bh < 0.1]
        goea_results_15   = [r for r in goea_results_15 if r.p_fdr_bh < 0.1]

        ## NB!! Uncomment to write down the results!!
        #goeaobj.wr_xlsx(f'./GOEAnalysis/GOATools_out/{deg_name}/GOATools_{deg_name}_{annotab.split("gene2go_")[-1]}_all.xlsx', goea_results_all)
        #goeaobj.wr_xlsx(f'./GOEAnalysis/GOATools_out/{deg_name}/GOATools_{deg_name}_{annotab.split("gene2go_")[-1]}_down.xlsx', goea_results_down)
        #goeaobj.wr_xlsx(f'./GOEAnalysis/GOATools_out/{deg_name}/GOATools_{deg_name}_{annotab.split("gene2go_")[-1]}_up.xlsx', goea_results_up)
        #goeaobj.wr_xlsx(f'./GOEAnalysis/GOATools_out/{deg_name}/GOATools_{deg_name}_{annotab.split("gene2go_")[-1]}_15.xlsx', goea_results_15)

        gc.collect()

In [ ]:
## Here I compared the GO annotations obtained with GO-basic.obo v.1.2 and v.1.4
## The differense was found to be magrinal and all interesting groups of genes
## remained, so there is no need to perform the same analysys with old GO-basic.obo
## ann1 = pd.read_table('./GOdb/Ldec_gene2go_ext2')[['GO_ID', 'GO_term']].groupby('GO_ID').first().reset_index()
## ann2 = pd.read_table('./GOdb/Ldec_gene2go_ncbi_ip_pi')[['GO_ID', 'GO_term']].groupby('GO_ID').first().reset_index()
## ann1[~ann1.GO_ID.isin( ann2.GO_ID )]

### 6.2 GOenrich


In [ ]:
## make directory GOenrich_out
## make subdirectories:
## - bbas_hae
## - mrob_hae
## - bbas_fat
## - mrob_fat
## - bbas_mrob_hae
## - bbas_mrob_fat
os.mkdir('./GOEAnalysis/GOenrich_out')
for oo in [ 'bbas_hae', 'mrob_hae', 'bbas_fat', 'mrob_fat', 'bbas_mrob_hae', 'bbas_mrob_fat' ]:
    os.mkdir(f'./GOEAnalysis/GOenrich_out/{oo}')


In [ ]:
O = goenrich.obo.ontology('./GOdb/go-basic-1.4.obo')
selected_bg = all_genes
background_attribute = 'selected_bg'

for annotab in ['Ldec_gene2go_ncbi_ip_pi', 'Ldec_gene2go_ncbi_pi', 'Ldec_gene2go_pi', 
                'Ldec_gene2go_ncbi_ip_pihq', 'Ldec_gene2go_ncbi_pihq', 'Ldec_gene2go_pihq']:
    gene2go = goenrich.read.gene2go(f'./GOdb/{annotab}', tax_id = 7539)
    values = {k: set(v) for k,v in gene2go.groupby('GO_ID')['GeneID']}
    goenrich.enrich.propagate(O, values, background_attribute)
    for deg_name in [ 'bbas_hae', 'mrob_hae', 'bbas_fat', 'mrob_fat', 'bbas_mrob_hae', 'bbas_mrob_fat' ]:
        deg_tab = globals()[ deg_name ]
        gene_query      = deg_tab[ (deg_tab.padj < 0.1) ].GeneID.values.tolist()
        gene_query_15   = deg_tab[ (deg_tab.log2FoldChange.abs() >= .5) & (deg_tab.padj < 0.1) ].GeneID.values.tolist()
        gene_query_up   = deg_tab[ (deg_tab.log2FoldChange >= .5) & (deg_tab.padj < 0.1) ].GeneID.values.tolist()
        gene_query_down = deg_tab[ (deg_tab.log2FoldChange <= -.5) & (deg_tab.padj < 0.1) ].GeneID.values.tolist()
        
        goea_results_all  = goenrich.enrich.analyze(O, gene_query, background_attribute) 
        goea_results_down = goenrich.enrich.analyze(O, gene_query_down, background_attribute) 
        goea_results_up   = goenrich.enrich.analyze(O, gene_query_up, background_attribute)
        goea_results_15   = goenrich.enrich.analyze(O, gene_query_15, background_attribute)

        goea_results_all  = goea_results_all [goea_results_all.q  < .1].sort_values(['namespace','p'])
        goea_results_down = goea_results_down[goea_results_down.q < .1].sort_values(['namespace','p'])
        goea_results_up   = goea_results_up  [goea_results_up.q   < .1].sort_values(['namespace','p'])
        goea_results_15   = goea_results_15  [goea_results_15.q   < .1].sort_values(['namespace','p'])

        ## NB!! Uncomment to write down the results!!
        #goea_results_all.to_excel(f'./GOEAnalysis/GOenrich_out/{deg_name}/GOenrich_{deg_name}_{annotab.split("gene2go_")[-1]}_all.xlsx')
        #goea_results_down.to_excel(f'./GOEAnalysis/GOenrich_out/{deg_name}/GOenrich_{deg_name}_{annotab.split("gene2go_")[-1]}_down.xlsx')
        #goea_results_up.to_excel(f'./GOEAnalysis/GOenrich_out/{deg_name}/GOenrich_{deg_name}_{annotab.split("gene2go_")[-1]}_up.xlsx')
        #goea_results_15.to_excel(f'./GOEAnalysis/GOenrich_out/{deg_name}/GOenrich_{deg_name}_{annotab.split("gene2go_")[-1]}_15.xlsx')

        gc.collect()

## 7. Add InterProScan and ProteInfer information to DEG tables


In [ ]:
ldec_pfams = protinfer[protinfer.predicted_label.str.startswith('Pfam:')].groupby(
    'GeneID')['description'].agg( lambda x: '; '.join(list(set(x))) ).reset_index().rename( columns = {'description':'PI_Pfam'} )

In [ ]:
ldec_pigos = protinfer[protinfer.predicted_label.str.startswith('GO:')].groupby(
    'GeneID')['description'].agg( lambda x: '; '.join(list(set(x))) ).reset_index().rename( columns = {'description':'PI_GO'} )

In [ ]:
ldec_ipgos = interpro[ interpro.Main_GO.str.startswith('GO:') & (
    interpro.Main_GO == interpro.All_GO) ][['GO_term', 'GeneID']].drop_duplicates().groupby(
    'GeneID')['GO_term'].agg( lambda x: '; '.join(list(set(x))) ).reset_index().rename( columns = {'GO_term':'IP_GO'} )

In [ ]:
ldec_genes_descriptions = ldec_pfams.merge( ldec_ipgos, how = 'outer' ).merge( ldec_pigos , how = 'outer' ).fillna('')

In [ ]:
writer = pd.ExcelWriter(f"./DEG_tables_all_annotated.xlsx", engine="openpyxl", mode="w")

for deg_name in [ 'bbas_hae', 'mrob_hae', 'bbas_fat', 'mrob_fat', 'bbas_mrob_hae', 'bbas_mrob_fat' ]:
    deg_tab = globals()[ deg_name ]
    deg_tab2 = deg_tab.merge( ldec_genes_descriptions )
    deg_tab2.to_excel(writer, sheet_name = deg_name)
writer.close()


In [ ]:
!wsl ls ./GOEAnalysis/GOATools_out/bbas_hae